In [1]:
import pandas as pd
import time
import os
import paramiko
from dotenv import load_dotenv
from pathlib import Path
import posixpath

#time.sleep(60)
remote_folder = "wikidata-id-person-fix"
local_folder = "wikidata-id-person-fix"
local_file_basename = "wikidata-id-person-fix"

# Load SFTP credentials from .env
load_dotenv(dotenv_path=Path(".env"), override=True)

sftp_host = os.getenv("SFTP_HOST")
sftp_login = os.getenv("SFTP_LOGIN")
sftp_password = os.getenv("SFTP_PASSWORD")
sftp_base_folder = os.getenv("SFTP_FOLDER")

if not sftp_host or not sftp_login or sftp_password is None or not sftp_base_folder:
    raise ValueError("Missing SFTP configuration in .env: SFTP_HOST, SFTP_LOGIN, SFTP_PASSWORD, SFTP_FOLDER")

remote_path = posixpath.join(sftp_base_folder, remote_folder)
local_path = Path("./data") / local_folder
local_path.mkdir(parents=True, exist_ok=True)

downloaded_count = 0
skipped_count = 0

transport = paramiko.Transport((sftp_host, 22))
transport.connect(username=sftp_login, password=sftp_password)
sftp = paramiko.SFTPClient.from_transport(transport)

try:
    for entry in sftp.listdir_attr(remote_path):
        if entry.filename in (".", ".."):  # defensive
            continue

        remote_file = posixpath.join(remote_path, entry.filename)
        local_file = local_path / entry.filename

        try:
            # Skip directories
            if (entry.st_mode & 0o170000) == 0o040000:
                continue
        except Exception:
            pass

        if local_file.exists():
            skipped_count += 1
            continue

        sftp.get(remote_file, str(local_file))
        downloaded_count += 1
finally:
    sftp.close()
    transport.close()

print(f"SFTP folder: {remote_path}")
print(f"Local folder: {local_path.resolve()}")
print(f"Downloaded {downloaded_count} new file(s), skipped {skipped_count} existing file(s)")


SFTP folder: /home/debian/docker/selenium-tmdb/wikidata-id-person-fix
Local folder: C:\Users\vaugo\Code\selenium-tmdb\data\wikidata-id-person-fix
Downloaded 0 new file(s), skipped 8 existing file(s)


In [2]:
#pip install selenium webdriver-manager
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

In [3]:
if 1:
    # Reading value
    %store -r lngpersonidstart
    print("lngpersonidstart = ",lngpersonidstart) 
    lngpersonidstart -= 1
    print("lngpersonidstart = ",lngpersonidstart) 


lngpersonidstart =  5669766
lngpersonidstart =  5669765


In [4]:
driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()))

In [5]:
driver.get("https://www.themoviedb.org/login")
time.sleep(10)

In [6]:
username_field = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.NAME, "username"))
)
username_field.clear()
username_field.send_keys(os.getenv("TMDB_LOGIN"))

password_field = driver.find_element(By.NAME, "password")
password_field.clear()
password_field.send_keys(os.getenv("TMDB_PASSWORD"))
password_field.send_keys("\n")


## Update persons

In [7]:
from pathlib import Path

# Prefer the downloaded SFTP folder if it exists, otherwise fall back to ./data
base_candidates = [Path("./data") / local_folder, Path("./data")]

csv_candidates = []
for base in base_candidates:
    if base.exists():
        csv_candidates.extend(base.glob(local_file_basename + "*.csv"))

if not csv_candidates:
    raise FileNotFoundError(
        "No file found matching "
        + local_file_basename
        + "*.csv in ./data or ./data/<local_folder>"
    )

# The export date lives in the filename; mtime is only the SFTP download
# time, identical across a whole mirror pass, so it cannot order the exports.
latest_csv = max(csv_candidates, key=lambda p: (p.name, str(p)))
print(f"Using latest CSV: {latest_csv}")

data = pd.read_csv(str(latest_csv), sep=';', quotechar='"')
intgoingdown = True
lngpersonidstart = 0


Using latest CSV: data\wikidata-id-person-fix\wikidata-id-person-fix-20260610.csv


In [8]:
data.shape

(241, 8)

In [9]:
def f_tmdbpersonsetwikidataid(lngid, strwikidataid):
    driver.get(f"https://www.themoviedb.org/person/{lngid}/edit?active_nav_item=external_ids")
    intpagefound = True
    try:
        elements = driver.find_elements(By.XPATH, "//h2[text()=\"Oops! We can't find the page you're looking for\"]")
        if len(elements) > 0:
            print("Element 'Page 404' exists on the page.")
            intpagefound = False
        else:
            print("Element 'Page 404' does not exist on the page.")
    except NoSuchElementException:
        print("Element 'Page 404' does not exist on the page.")
    if intpagefound:
        # Define the XPath for the button containing the specific span text
        xpath = "//button[span[contains(@class, 'glyphicons_v2') and contains(@class, 'plus') and contains(@class, 'svg')] and contains(., 'Create Translation')]"
        try:
            # Locate the button using XPath
            button = driver.find_element(By.XPATH, xpath)
            # Click the button
            button.click()
            print("Button 'Create translation' clicked successfully.")
            time.sleep(2)
            driver.get(f"https://www.themoviedb.org/person/{lngid}/edit?active_nav_item=external_ids")
        except Exception as e:
            print("Button 'Create translation' not found, translation already exists.")
        try:
            wikidata_id_field = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.NAME, "wikidata_id"))
            )
        except TimeoutException:
            print(f"wikidata_id field not found for person {lngid}. Skipping.")
            return
        # Is the record locked? 
        css_selector = "span#wikidata_id_status.glyphicons_v2.locked.locked_status"
        elements = driver.find_elements(By.CSS_SELECTOR, css_selector)

        # Check if the element exists by verifying the list is not empty
        if len(elements) > 0:
            # Record is locked
            print("Element 'Locked' exists on the page.")
        else:
            # Record is not locked
            wikidata_id_field.clear()
            wikidata_id_field.send_keys(strwikidataid)

            #save_button = driver.find_element(By.XPATH, "//button[@type='submit']")
            save_button = driver.find_element(By.XPATH, '//input[@value="Save"]')
            save_button.click()

In [10]:
def f_tmdbpersonclearwikidataid(lngid):
    driver.get(f"https://www.themoviedb.org/person/{lngid}/edit?active_nav_item=external_ids")
    wikidata_id_field = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.NAME, "wikidata_id"))
    )
    # Is the record locked?
    css_selector = "span#wikidata_id_status.glyphicons_v2.locked.locked_status"
    elements = driver.find_elements(By.CSS_SELECTOR, css_selector)
    if len(elements) > 0:
        print("Element 'Locked' exists on the page. Skipping clear.")
        return

    wikidata_id_field.clear()

    #save_button = driver.find_element(By.XPATH, "//button[@type='submit']")
    save_button = driver.find_element(By.XPATH, '//input[@value="Save"]')
    save_button.click()

In [11]:
lngprocessedcount = 0
if intgoingdown:
    # Going from the top to the end of the dataframe 
    for index, row in data.iterrows():
        print(f"Index: {index} ; processed: {lngprocessedcount}")
        if row['ID_PERSON']:
            lngid = row['ID_PERSON']
            if row['ID_WIKIDATA']:
                strwikidataid = row['ID_WIKIDATA']
                strtmdbidtoerase = row['ID_PERSON_ERASE_WIKIDATA_ID']
                if lngid > lngpersonidstart:
                    # Check if strtmdbidtoerase is nan
                    if pd.isna(strtmdbidtoerase):
                        print("strtmdbidtoerase is NaN")
                    else:
                        print("strtmdbidtoerase is not NaN")
                        # Convert to int
                        lngtmdbidtoerase = int(strtmdbidtoerase)
                        f_tmdbpersonclearwikidataid(lngtmdbidtoerase)
                    #print("strtmdbidtoerase",strtmdbidtoerase)
                    f_tmdbpersonsetwikidataid(lngid,strwikidataid)
                    lngprocessedcount += 1
                    #f_tmdbpersonclearwikidataid(lngid)
                    lngpersonidstart = lngid
                    %store lngpersonidstart
                    time.sleep(2)
            else:
                print("Colonne ID_WIKIDATA manquante dans le fichier CSV")
        else:
            print("Colonne ID_PERSON manquante dans le fichier CSV")
else:
    # Going from the end to the top of the dataframe 
    for index, row in data.iloc[::-1].iterrows():
        print(f"Index: {index} ; processed: {lngprocessedcount}")
        if row['ID_PERSON']:
            lngid = row['ID_PERSON']
            if row['ID_WIKIDATA']:
                strwikidataid = row['ID_WIKIDATA']
                strtmdbidtoerase = row['ID_PERSON_ERASE_WIKIDATA_ID']
                if lngid < lngpersonidstart:
                    # Check if strtmdbidtoerase is nan
                    if pd.isna(strtmdbidtoerase):
                        print("strtmdbidtoerase is NaN")
                    else:
                        print("strtmdbidtoerase is not NaN")
                        # Convert to int
                        lngtmdbidtoerase = int(strtmdbidtoerase)
                        f_tmdbpersonclearwikidataid(lngtmdbidtoerase)
                    #print("strtmdbidtoerase",strtmdbidtoerase)
                    f_tmdbpersonsetwikidataid(lngid,strwikidataid)
                    lngprocessedcount += 1
                    #f_tmdbpersonclearwikidataid(lngid)
                    lngpersonidstart = lngid
                    %store lngpersonidstart
                    time.sleep(2)
            else:
                print("Colonne ID_WIKIDATA manquante dans le fichier CSV")
        else:
            print("Colonne ID_PERSON manquante dans le fichier CSV")


Index: 0 ; processed: 0
strtmdbidtoerase is NaN
Element 'Page 404' does not exist on the page.
Button 'Create translation' not found, translation already exists.
Element 'Locked' exists on the page.
Stored 'lngpersonidstart' (int)
Index: 1 ; processed: 1
strtmdbidtoerase is NaN
Element 'Page 404' does not exist on the page.
Button 'Create translation' not found, translation already exists.
Stored 'lngpersonidstart' (int)
Index: 2 ; processed: 2
strtmdbidtoerase is not NaN
Element 'Locked' exists on the page. Skipping clear.
Element 'Page 404' does not exist on the page.
Button 'Create translation' not found, translation already exists.
Stored 'lngpersonidstart' (int)
Index: 3 ; processed: 3
strtmdbidtoerase is NaN
Element 'Page 404' does not exist on the page.
Button 'Create translation' not found, translation already exists.
Stored 'lngpersonidstart' (int)
Index: 4 ; processed: 4
strtmdbidtoerase is NaN
Element 'Page 404' does not exist on the page.
Button 'Create translation' not fou

In [12]:
if 0:
    # Reading value
    %store -r lngpersonidstart
    print("lngpersonidstart = ",lngpersonidstart) 

In [13]:
print("lngpersonidstart = ",lngpersonidstart) 

lngpersonidstart =  6267046


In [14]:
# Loop is finished so we display the home page
driver.get(f"https://www.themoviedb.org/")